# FINet evaluation on real unseen data
Use actual completed checkpoints. This notebook does not contain experimental results. It excludes development-image overlap and records geometry exclusions. Protocol: raw probabilities, S-measure without min-max scaling, native MAE, binary threshold 0.5 and Boundary F1 with a two-pixel native-resolution tolerance. Both models use the same images and settings.

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/FINet')
archive = Path('/content/finet_colab_training.zip')
if not archive.exists():
    archive = DRIVE_ROOT / 'code/finet_colab_training.zip'
assert archive.is_file(), 'Upload the current code ZIP or restore its Drive backup.'
PROJECT = Path('/content/boundary-aware-finet')
with zipfile.ZipFile(archive) as zipped:
    for name in zipped.namelist():
        assert not name.startswith('/') and '..' not in Path(name).parts
    zipped.extractall(PROJECT)
os.chdir(PROJECT)
def run_checked(command):
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout); print(completed.stderr[-5000:]); completed.check_returncode()
run_checked([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-evaluation.txt', '-r', 'requirements-data.txt'])
import torch
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print({'device': DEVICE, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

In [ ]:
BASELINE = DRIVE_ROOT / 'experiments/baseline_seed123456_b4_v1/best.pth'
BOUNDARY = DRIVE_ROOT / 'experiments/boundary_seed123456_b4_v1/best.pth'
SPLIT = DRIVE_ROOT / 'splits/cod_camo_seed123456.json'
TEST_ROOT = Path('/content/finet_test_data')
TEST_ARCHIVE = Path('/content/COD-TestDataset.zip')
DATASET = 'CAMO'  # Also CHAMELEON, COD10K or NC4K; preserve each dataset's report.
OUTPUT = DRIVE_ROOT / 'evaluation/camo_comparison_v1'
print({'baseline_exists': BASELINE.is_file(), 'boundary_exists': BOUNDARY.is_file(), 'output':str(OUTPUT)})

In [ ]:
run_checked([sys.executable, 'scripts/prepare_test_data.py', '--archive', str(TEST_ARCHIVE), '--data-root', str(TEST_ROOT), '--training-manifest', str(SPLIT), '--report', str(DRIVE_ROOT/'evaluation/test_data_audit.json'), '--download'])

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    assert BASELINE.is_file(), 'No trained baseline checkpoint.'
    command = [sys.executable, 'evaluate_models.py', '--baseline', str(BASELINE), '--dataset', str(TEST_ROOT/DATASET), '--training-manifest', str(SPLIT), '--output', str(OUTPUT), '--device', DEVICE, '--threshold', '0.5', '--boundary-tolerance', '2']
    if BOUNDARY.is_file():
        command += ['--boundary', str(BOUNDARY)]
    run_checked(command)
else:
    print('Evaluation not launched. Confirm genuine trained checkpoints and choose a fresh output directory.')